In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/input/datasets/vtphatt2")

DATASETS = [
    "genimage-adm",
    "genimage-biggan",
    "genimage-glide",
    "genimage-midjourney-part-1",
    "genimage-stable-diffusion-v1-4",
    "genimage-stable-diffusion-v1-5",
    "genimage-wukong",
]

def show_tree(root, max_depth=6, depth=0):
    if depth > max_depth:
        return

    try:
        entries = sorted(root.iterdir(), key=lambda x: x.name.lower())
    except Exception as e:
        print("  ERROR:", e)
        return

    for p in entries:
        indent = "    " * depth

        if p.is_dir():
            print(f"{indent}📁 {p.name}/")

            # If this is a potential image directory, DON'T enter it.
            if p.name.lower() in {"ai", "nature", "real", "fake"}:
                try:
                    # This operation reads directory metadata only.
                    count = sum(1 for _ in p.iterdir())
                    print(f"{indent}   → {count:,} files")
                except Exception as e:
                    print(f"{indent}   → unable to count: {e}")
            else:
                show_tree(p, max_depth, depth + 1)

        else:
            print(f"{indent}📄 {p.name}")


for dataset in DATASETS:
    path = ROOT / dataset

    print("\n" + "=" * 90)
    print(dataset)
    print("=" * 90)

    if not path.exists():
        print("❌ NOT FOUND")
        continue

    show_tree(path)

In [ ]:
from pathlib import Path
from collections import Counter
from PIL import Image
import pandas as pd
import random
import json
import os

ROOT = Path("/kaggle/input/datasets/vtphatt2")
OUT = Path("/kaggle/working/signalscope_audit")
OUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Exact dataset locations discovered from Step 1
# ------------------------------------------------------------

DATASETS = {
    "ADM": ROOT / "genimage-adm/GenImage/ADM/imagenet_ai_0508_adm",
    "BigGAN": ROOT / "genimage-biggan/BigGAN/imagenet_ai_0419_biggan",
    "GLIDE": ROOT / "genimage-glide/glide/imagenet_glide",
    "SD1.4": ROOT / "genimage-stable-diffusion-v1-4/GenImage/stable_diffusion_v_1_4",
    "SD1.5": ROOT / "genimage-stable-diffusion-v1-5/GenImage/stable_diffusion_v_1_5",
    "Wukong": ROOT / "genimage-wukong/GenImage/wukong",
}

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# Number of images to actually open per class/split.
# This is enough for an initial statistical audit.
SAMPLE_PER_FOLDER = 300

random.seed(42)

summary = []
samples = []

print("=" * 100)
print("SIGNALSCOPE — STEP 2 DATASET AUDIT")
print("=" * 100)

# ------------------------------------------------------------
# Audit normal directory-based datasets
# ------------------------------------------------------------

for generator, base in DATASETS.items():

    print(f"\n{'=' * 80}")
    print(f"GENERATOR: {generator}")
    print(f"PATH: {base}")
    print("=" * 80)

    if not base.exists():
        print("❌ PATH NOT FOUND")
        continue

    for split in ["train", "val"]:
        for label_name, label in [("ai", 1), ("nature", 0)]:

            folder = base / split / label_name

            if not folder.exists():
                print(f"⚠️ Missing: {split}/{label_name}")
                continue

            # Directory listing only — no image decoding here
            files = [
                p for p in folder.iterdir()
                if p.is_file() and p.suffix.lower() in IMAGE_EXTS
            ]

            total = len(files)

            print(
                f"{split:5s} | "
                f"{label_name:6s} | "
                f"{total:,} images"
            )

            # Random sample for actual image inspection
            sample_files = (
                random.sample(files, min(SAMPLE_PER_FOLDER, total))
                if total > SAMPLE_PER_FOLDER
                else files
            )

            format_counts = Counter()
            size_counts = Counter()
            corrupted = 0
            exif_present = 0

            for path in sample_files:

                try:
                    with Image.open(path) as img:

                        # Verify image integrity
                        img.verify()

                    # Reopen because verify() closes image state
                    with Image.open(path) as img:

                        width, height = img.size

                        format_counts[path.suffix.lower()] += 1
                        size_counts[(width, height)] += 1

                        if img.getexif():
                            exif_present += 1

                        samples.append({
                            "generator": generator,
                            "split": split,
                            "label": label,
                            "label_name": label_name,
                            "path": str(path),
                            "width": width,
                            "height": height,
                            "format": path.suffix.lower(),
                            "aspect_ratio": round(width / height, 4)
                        })

                except Exception:
                    corrupted += 1

            summary.append({
                "generator": generator,
                "split": split,
                "label": label_name,
                "total_images": total,
                "sampled": len(sample_files),
                "corrupted_in_sample": corrupted,
                "exif_in_sample": exif_present,
                "formats": dict(format_counts),
                "top_resolutions": dict(size_counts.most_common(10)),
            })

# ------------------------------------------------------------
# Print summary
# ------------------------------------------------------------

df_summary = pd.DataFrame(summary)

print("\n\n" + "=" * 100)
print("IMAGE COUNTS")
print("=" * 100)

print(
    df_summary[
        ["generator", "split", "label", "total_images",
         "sampled", "corrupted_in_sample", "exif_in_sample"]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# Aggregate real/fake counts
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("REAL vs AI")
print("=" * 100)

counts = (
    df_summary
    .groupby(["generator", "split", "label"])["total_images"]
    .sum()
    .unstack(fill_value=0)
)

print(counts)

# ------------------------------------------------------------
# Sample-level format/resolution analysis
# ------------------------------------------------------------

df_samples = pd.DataFrame(samples)

print("\n" + "=" * 100)
print("SAMPLE FORMAT DISTRIBUTION")
print("=" * 100)

print(
    pd.crosstab(
        [df_samples.generator, df_samples.split, df_samples.label_name],
        df_samples.format
    )
)

print("\n" + "=" * 100)
print("TOP RESOLUTIONS FROM SAMPLE")
print("=" * 100)

resolution_table = (
    df_samples
    .groupby(["generator", "split", "label_name"])
    .apply(lambda x: x[["width", "height"]].value_counts().head(10), include_groups=False)
)

print(resolution_table)

print("\n" + "=" * 100)
print("ASPECT RATIO SUMMARY")
print("=" * 100)

print(
    df_samples
    .groupby(["generator", "split", "label_name"])["aspect_ratio"]
    .agg(["min", "median", "max"])
    .round(3)
)

# ------------------------------------------------------------
# Save audit results
# ------------------------------------------------------------

df_summary.to_csv(OUT / "audit_summary.csv", index=False)
df_samples.to_csv(OUT / "image_samples.csv", index=False)

with open(OUT / "audit_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 100)
print("AUDIT FILES SAVED")
print("=" * 100)

print(OUT / "audit_summary.csv")
print(OUT / "audit_summary.json")
print(OUT / "image_samples.csv")

print("\n✅ STEP 2 INITIAL AUDIT COMPLETE")

In [ ]:
# Fix JSON serialization of tuple resolution keys

def make_json_safe(obj):
    if isinstance(obj, dict):
        return {
            str(k): make_json_safe(v)
            for k, v in obj.items()
        }
    elif isinstance(obj, (list, tuple)):
        return [make_json_safe(v) for v in obj]
    else:
        return obj


safe_summary = make_json_safe(summary)

with open(OUT / "audit_summary.json", "w") as f:
    json.dump(safe_summary, f, indent=2)

print("✅ audit_summary.json saved")
print(f"📁 {OUT}")

In [ ]:
import pandas as pd
from pathlib import Path

AUDIT_DIR = Path("/kaggle/working/signalscope_audit")

df = pd.read_csv(AUDIT_DIR / "image_samples.csv")

print("=" * 90)
print("SIGNALSCOPE — STEP 3: DATASET BIAS ANALYSIS")
print("=" * 90)

# ---------------------------------------------------------
# 1. Format vs label
# ---------------------------------------------------------

print("\n[1] FORMAT × LABEL")
print("-" * 60)

print(
    pd.crosstab(
        df["label_name"],
        df["format"],
        normalize="index"
    ).round(3)
)

# ---------------------------------------------------------
# 2. Resolution vs label
# ---------------------------------------------------------

print("\n[2] RESOLUTION × LABEL")
print("-" * 60)

resolution = (
    df.groupby(["generator", "label_name"])
      .apply(lambda x: x[["width", "height"]]
             .value_counts()
             .head(5))
)

print(resolution)

# ---------------------------------------------------------
# 3. Aspect ratio
# ---------------------------------------------------------

print("\n[3] ASPECT RATIO")
print("-" * 60)

print(
    df.groupby(["generator", "label_name"])["aspect_ratio"]
      .agg(["min", "median", "max"])
      .round(3)
)

# ---------------------------------------------------------
# 4. EXIF
# ---------------------------------------------------------

print("\n[4] EXIF PRESENCE")
print("-" * 60)

print(
    df.groupby(["generator", "label_name"])["path"]
      .count()
)

# ---------------------------------------------------------
# 5. Potential shortcut report
# ---------------------------------------------------------

print("\n[5] SHORTCUT CHECK")
print("-" * 60)

format_by_label = (
    df.groupby(["label_name", "format"])
      .size()
      .unstack(fill_value=0)
)

print(format_by_label)

print("\n⚠️ IMPORTANT:")

if set(df[df.label_name == "ai"]["format"].unique()) != \
   set(df[df.label_name == "nature"]["format"].unique()):

    print("❌ FORMAT DISTRIBUTIONS DIFFER STRONGLY.")
    print("   We MUST neutralize format/compression shortcuts.")

print("\nResolution ranges:")

print(
    df.groupby("label_name")[["width", "height"]]
      .agg(["min", "median", "max"])
)

print("\n" + "=" * 90)
print("BIAS ANALYSIS COMPLETE")
print("=" * 90)

In [ ]:
from pathlib import Path
import os

ROOT = Path("/kaggle/input/datasets/vtphatt2")
MJ = ROOT / "genimage-midjourney-part-1" / "part_aa"

print("Path:", MJ)
print("Exists:", MJ.exists())
print("Size GB:", round(MJ.stat().st_size / (1024**3), 2) if MJ.exists() else None)

# Inspect file type/header only
with open(MJ, "rb") as f:
    header = f.read(32)

print("Header:", header)

In [ ]:
!file "/kaggle/input/datasets/vtphatt2/genimage-midjourney-part-1/part_aa"

In [ ]:
!unzip -l "/kaggle/input/datasets/vtphatt2/genimage-midjourney-part-1/part_aa" 2>&1 | head -30

In [ ]:
from pathlib import Path
import pandas as pd
import time

START = time.time()

print("=" * 90)
print("SIGNALSCOPE — STEP 5: BUILD RAW DATASET MANIFEST")
print("=" * 90)

ROOT = Path("/kaggle/input/datasets/vtphatt2")

DATASETS = {
    "ADM": ROOT / "genimage-adm/GenImage/ADM/imagenet_ai_0508_adm/train",
    "BigGAN": ROOT / "genimage-biggan/BigGAN/imagenet_ai_0419_biggan/train",
    "GLIDE": ROOT / "genimage-glide/glide/imagenet_glide/train",
    "SD1.4": ROOT / "genimage-stable-diffusion-v1-4/GenImage/stable_diffusion_v_1_4/train",
    "SD1.5": ROOT / "genimage-stable-diffusion-v1-5/GenImage/stable_diffusion_v_1_5/train",
    "Wukong": ROOT / "genimage-wukong/GenImage/wukong/train",
}

ROWS = []

print("\n[START] Scanning dataset directories...\n")

for generator, base in DATASETS.items():

    print(f"▶ {generator}")
    generator_start = time.time()

    for label in ["ai", "nature"]:

        folder = base / label
        print(f"   ├─ Scanning {label}: {folder}")

        count = 0

        for p in folder.iterdir():
            if p.is_file():
                ROWS.append({
                    "path": str(p),
                    "generator": generator,
                    "label": 1 if label == "ai" else 0,
                    "label_name": label,
                })

                count += 1

                # Progress every 10,000 files
                if count % 10_000 == 0:
                    elapsed = time.time() - START
                    print(
                        f"   │  ⏳ {label}: {count:,} files | "
                        f"Total collected: {len(ROWS):,} | "
                        f"Elapsed: {elapsed:.1f}s"
                    )

        print(f"   └─ ✅ {label}: {count:,} files")

    print(
        f"   ✅ {generator} complete "
        f"({time.time() - generator_start:.1f}s)\n"
    )

print("=" * 90)
print("BUILDING DATAFRAME")
print("=" * 90)

df = pd.DataFrame(ROWS)

print(f"\n✅ DataFrame created")
print(f"Total images: {len(df):,}")

print("\nGenerator × Label:")
print(pd.crosstab(df["generator"], df["label_name"]))

print("\nTotal AI:", f"{(df['label'] == 1).sum():,}")
print("Total Real:", f"{(df['label'] == 0).sum():,}")

print("\n" + "=" * 90)
print("SAVING MANIFEST")
print("=" * 90)

OUT = Path("/kaggle/working/signalscope_audit")
OUT.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUT / "raw_manifest.csv"

print(f"💾 Saving to:\n{OUTPUT_FILE}")

df.to_csv(OUTPUT_FILE, index=False)

print("\n✅ Manifest saved successfully!")
print(f"📄 File size: {OUTPUT_FILE.stat().st_size / (1024**2):.2f} MB")
print(f"⏱️ Total execution time: {time.time() - START:.1f} seconds")

print("\n" + "=" * 90)
print("STEP 5 COMPLETE")
print("=" * 90)

In [9]:
from pathlib import Path
import pandas as pd
import time

START = time.time()

print("=" * 90)
print("SIGNALSCOPE — STEP 5B: BUILD CANDIDATE SUBSET")
print("=" * 90)

MANIFEST = Path("/kaggle/working/signalscope_audit/raw_manifest.csv")
OUT = Path("/kaggle/working/signalscope")
OUT.mkdir(parents=True, exist_ok=True)

if not MANIFEST.exists():
    raise FileNotFoundError(
        "raw_manifest.csv is missing. STOP here — do NOT rescan 1.9M files yet."
    )

print("\n[1/5] Loading raw manifest...")
df = pd.read_csv(MANIFEST)

print(f"✅ Loaded {len(df):,} rows")
print(f"⏱️ Elapsed: {time.time() - START:.1f}s")

# ---------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------

SEED = 42
AI_PER_GENERATOR = 5000
TOTAL_REAL = 30000

GENERATORS = [
    "ADM",
    "BigGAN",
    "GLIDE",
    "SD1.4",
    "SD1.5",
    "Wukong",
]

# ---------------------------------------------------------
# AI SAMPLE
# ---------------------------------------------------------

print("\n[2/5] Sampling AI images...")

ai_parts = []

for generator in GENERATORS:

    print(f"   ⏳ Sampling {generator}...")

    part = df[
        (df["generator"] == generator) &
        (df["label"] == 1)
    ]

    sampled = part.sample(
        n=AI_PER_GENERATOR,
        random_state=SEED
    )

    ai_parts.append(sampled)

    print(
        f"   ✅ {generator}: {len(sampled):,} selected "
        f"| available={len(part):,}"
    )

ai_df = pd.concat(ai_parts, ignore_index=True)

print(f"\n✅ Total AI selected: {len(ai_df):,}")

# ---------------------------------------------------------
# REAL SAMPLE
# ---------------------------------------------------------

print("\n[3/5] Preparing real-image pool...")

real_df = df[df["label"] == 0].copy()

print(f"   Raw real rows: {len(real_df):,}")

# GenImage may reuse the same ImageNet real image across generator folders.
# First deduplicate using filename.
real_df["filename"] = real_df["path"].map(
    lambda x: Path(x).name
)

before = len(real_df)

real_unique = real_df.drop_duplicates(
    subset=["filename"],
    keep="first"
)

print(f"   After filename dedup: {len(real_unique):,}")
print(f"   Potential repeated filenames removed: {before - len(real_unique):,}")

if len(real_unique) < TOTAL_REAL:
    raise RuntimeError(
        f"Only {len(real_unique):,} unique real filenames available."
    )

print("\n   ⏳ Sampling real images...")

real_sample = real_unique.sample(
    n=TOTAL_REAL,
    random_state=SEED
)

print(f"✅ Real selected: {len(real_sample):,}")

# ---------------------------------------------------------
# COMBINE
# ---------------------------------------------------------

print("\n[4/5] Combining candidate dataset...")

candidate = pd.concat(
    [ai_df, real_sample],
    ignore_index=True
)

candidate = candidate.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

print(f"✅ Total candidate images: {len(candidate):,}")

print("\nClass distribution:")
print(candidate["label"].value_counts())

print("\nAI generator distribution:")
print(
    candidate[candidate["label"] == 1]
    ["generator"]
    .value_counts()
)

# ---------------------------------------------------------
# SAVE
# ---------------------------------------------------------

print("\n[5/5] Saving candidate manifest...")

output_file = OUT / "candidate_manifest_60k.csv"

candidate.to_csv(
    output_file,
    index=False
)

print(f"\n✅ Saved:")
print(output_file)

print(
    f"📄 Size: "
    f"{output_file.stat().st_size / (1024**2):.2f} MB"
)

print(
    f"⏱️ Total runtime: "
    f"{time.time() - START:.1f}s"
)

print("\n" + "=" * 90)
print("STEP 5B COMPLETE")
print("=" * 90)

SIGNALSCOPE — STEP 5B: BUILD CANDIDATE SUBSET

[1/5] Loading raw manifest...
✅ Loaded 1,933,463 rows
⏱️ Elapsed: 5.4s

[2/5] Sampling AI images...
   ⏳ Sampling ADM...
   ✅ ADM: 5,000 selected | available=162,000
   ⏳ Sampling BigGAN...
   ✅ BigGAN: 5,000 selected | available=162,000
   ⏳ Sampling GLIDE...
   ✅ GLIDE: 5,000 selected | available=162,000
   ⏳ Sampling SD1.4...
   ✅ SD1.4: 5,000 selected | available=161,997
   ⏳ Sampling SD1.5...
   ✅ SD1.5: 5,000 selected | available=166,000
   ⏳ Sampling Wukong...
   ✅ Wukong: 5,000 selected | available=162,000

✅ Total AI selected: 30,000

[3/5] Preparing real-image pool...
   Raw real rows: 957,466
   After filename dedup: 957,466
   Potential repeated filenames removed: 0

   ⏳ Sampling real images...
✅ Real selected: 30,000

[4/5] Combining candidate dataset...
✅ Total candidate images: 60,000

Class distribution:
label
1    30000
0    30000
Name: count, dtype: int64

AI generator distribution:
generator
GLIDE     5000
BigGAN    500

In [ ]:
from pathlib import Path
from PIL import Image
import pandas as pd
import hashlib
import time

START = time.time()

print("=" * 90)
print("SIGNALSCOPE — STEP 6: VERIFY CANDIDATE DATASET")
print("=" * 90)

MANIFEST = Path("/kaggle/working/signalscope/candidate_manifest_60k.csv")
OUT = Path("/kaggle/working/signalscope")
OUTPUT = OUT / "verified_manifest_60k.csv"

print("\n[1/4] Loading candidate manifest...")

df = pd.read_csv(MANIFEST)

print(f"✅ Loaded {len(df):,} rows")
print(f"⏱️ Elapsed: {time.time() - START:.1f}s")

# ---------------------------------------------------------
# HASH + CORRUPTION CHECK
# ---------------------------------------------------------

print("\n[2/4] Checking files + computing SHA256...")
print("This opens each of the 60,000 selected images once.\n")

results = []

total = len(df)

for idx, row in df.iterrows():

    path = Path(row["path"])

    status = "ok"
    sha256 = None
    width = None
    height = None

    try:
        # -----------------------------------------
        # Verify image readability
        # -----------------------------------------
        with Image.open(path) as img:
            width, height = img.size
            img.verify()

        # -----------------------------------------
        # SHA256 exact-content hash
        # -----------------------------------------
        h = hashlib.sha256()

        with open(path, "rb") as f:
            while True:
                chunk = f.read(1024 * 1024)

                if not chunk:
                    break

                h.update(chunk)

        sha256 = h.hexdigest()

    except Exception as e:
        status = f"corrupt:{type(e).__name__}"

    results.append({
        "sha256": sha256,
        "verify_status": status,
        "width": width,
        "height": height,
    })

    # Progress every 1,000 images
    done = idx + 1

    if done % 1000 == 0 or done == total:

        elapsed = time.time() - START
        rate = done / elapsed if elapsed > 0 else 0

        print(
            f"⏳ {done:,}/{total:,} checked "
            f"({done/total*100:.1f}%) | "
            f"{rate:.1f} images/sec | "
            f"Elapsed: {elapsed/60:.1f} min"
        )

# ---------------------------------------------------------
# ATTACH RESULTS
# ---------------------------------------------------------

print("\n[3/4] Analysing verification results...")

result_df = pd.DataFrame(results)

df = pd.concat(
    [df.reset_index(drop=True), result_df],
    axis=1
)

corrupt = df[df["verify_status"] != "ok"]

print(f"\nCorrupted/unreadable: {len(corrupt):,}")

# ---------------------------------------------------------
# EXACT DUPLICATES
# ---------------------------------------------------------

valid = df[
    (df["verify_status"] == "ok") &
    (df["sha256"].notna())
].copy()

duplicate_mask = valid.duplicated(
    subset=["sha256"],
    keep="first"
)

duplicates = valid[duplicate_mask]

print(f"Exact duplicate rows: {len(duplicates):,}")

# Number of hashes occurring multiple times
duplicate_groups = (
    valid.groupby("sha256")
    .size()
    .sort_values(ascending=False)
)

duplicate_groups = duplicate_groups[
    duplicate_groups > 1
]

print(f"Duplicate hash groups: {len(duplicate_groups):,}")

# ---------------------------------------------------------
# CHECK LABEL COLLISIONS
# ---------------------------------------------------------

collision = (
    valid.groupby("sha256")["label"]
    .nunique()
)

collision = collision[
    collision > 1
]

print(f"⚠️ Same image appearing as BOTH AI and real: {len(collision):,}")

# ---------------------------------------------------------
# CLEAN
# ---------------------------------------------------------

print("\n[4/4] Creating verified manifest...")

clean = valid.drop_duplicates(
    subset=["sha256"],
    keep="first"
).copy()

clean.reset_index(drop=True, inplace=True)

clean.to_csv(
    OUTPUT,
    index=False
)

print("\nFinal counts:")
print(clean["label"].value_counts())

print("\nAI generators:")
print(
    clean[clean["label"] == 1]
    ["generator"]
    .value_counts()
)

print("\n✅ Saved:")
print(OUTPUT)

print(
    f"📄 Size: "
    f"{OUTPUT.stat().st_size / (1024**2):.2f} MB"
)

print(
    f"⏱️ Total runtime: "
    f"{(time.time() - START)/60:.1f} minutes"
)

print("\n" + "=" * 90)
print("STEP 6 COMPLETE")
print("=" * 90)

SIGNALSCOPE — STEP 6: VERIFY CANDIDATE DATASET

[1/4] Loading candidate manifest...
✅ Loaded 60,000 rows
⏱️ Elapsed: 0.2s

[2/4] Checking files + computing SHA256...
This opens each of the 60,000 selected images once.

⏳ 1,000/60,000 checked (1.7%) | 86.8 images/sec | Elapsed: 0.2 min
⏳ 2,000/60,000 checked (3.3%) | 90.2 images/sec | Elapsed: 0.4 min
⏳ 3,000/60,000 checked (5.0%) | 91.9 images/sec | Elapsed: 0.5 min
⏳ 4,000/60,000 checked (6.7%) | 92.3 images/sec | Elapsed: 0.7 min
⏳ 5,000/60,000 checked (8.3%) | 92.2 images/sec | Elapsed: 0.9 min
⏳ 6,000/60,000 checked (10.0%) | 92.2 images/sec | Elapsed: 1.1 min
⏳ 7,000/60,000 checked (11.7%) | 92.0 images/sec | Elapsed: 1.3 min
⏳ 8,000/60,000 checked (13.3%) | 91.4 images/sec | Elapsed: 1.5 min
⏳ 9,000/60,000 checked (15.0%) | 89.4 images/sec | Elapsed: 1.7 min
⏳ 10,000/60,000 checked (16.7%) | 88.3 images/sec | Elapsed: 1.9 min
⏳ 11,000/60,000 checked (18.3%) | 87.8 images/sec | Elapsed: 2.1 min
⏳ 12,000/60,000 checked (20.0%) | 85